In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================
input_customer = '../data/interim/customer_base.csv'
input_trans = '../data/interim/transactions_cleaned.csv'
output_path = '../data/processed/customer_features.csv'

print("Đang đọc dữ liệu...")
df_customer = pd.read_csv(input_customer)
df_trans = pd.read_csv(input_trans)
print("Đọc dữ liệu thành công!")

# ==========================================
# 2. TÍNH CÁC BIẾN BỊ THIẾU (UniqueProducts & Country)
# ==========================================
# File customer_base.csv của Nguyên Khôi thiếu cột UniqueProducts và Country
# Ta sẽ tính từ file transactions_cleaned.csv
df_customer['Customer ID'] = df_customer['Customer ID'].astype(str)
df_trans['Customer ID'] = df_trans['Customer ID'].astype(str)

customer_extras = df_trans.groupby('Customer ID').agg(
    UniqueProducts=('StockCode', 'nunique'),
    Country=('Country', 'first')
).reset_index()

# Merge vào bảng customer_base
df_customer = df_customer.merge(customer_extras, on='Customer ID', how='left')
df_customer['UniqueProducts'] = df_customer['UniqueProducts'].fillna(0)
print("Đã lấy thêm cột UniqueProducts và Country.")

# ==========================================
# 3. TÍNH TOÁN RFM VÀ CÁC BIẾN BỔ SUNG
# ==========================================
# Chuyển đổi ngày tháng (chú ý tên cột có dấu gạch dưới _)
df_customer['Last_Purchase_Date'] = pd.to_datetime(df_customer['Last_Purchase_Date'], errors='coerce')

# Ngày tham chiếu: Ngày cuối cùng của dữ liệu + 1 ngày (10/12/2011)
reference_date = pd.to_datetime('2011-12-10')

# Recency: Số ngày kể từ lần mua cuối
df_customer['Recency'] = (reference_date - df_customer['Last_Purchase_Date']).dt.days

# Frequency: Số hóa đơn (dùng cột Total_Invoices)
df_customer['Frequency'] = df_customer['Total_Invoices']

# Monetary: Tổng chi tiêu (dùng cột Total_Spend)
df_customer['Monetary'] = df_customer['Total_Spend']

# AvgOrderValue: Giá trị trung bình mỗi hóa đơn
df_customer['AvgOrderValue'] = df_customer['Monetary'] / df_customer['Frequency']
df_customer['AvgOrderValue'] = df_customer['AvgOrderValue'].replace([np.inf, -np.inf], 0).fillna(0)

# TotalItems (dùng cột Total_Items)
df_customer['TotalItems'] = df_customer['Total_Items']

print("Đã tính toán xong RFM và các biến bổ sung.")

# ==========================================
# 4. KIỂM TRA VÀ LƯU OUTPUT
# ==========================================
# Định dạng số thập phân (làm tròn 2 chữ số)
pd.options.display.float_format = '{:,.2f}'.format

# Chọn các cột cần hiển thị
cols_to_show = ['Customer ID', 'Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'TotalItems', 'Country']

# Lấy 5 dòng đầu và CHUYỂN VỊ (thêm .T ở cuối)
print("Xem trước dữ liệu (đã chuyển vị):")
print(df_customer[cols_to_show].head().T)

# Tạo thư mục nếu chưa có và lưu file
os.makedirs('../data/processed', exist_ok=True)
df_customer.to_csv(output_path, index=False)
print(f"\nĐã lưu {output_path} thành công!")

Đang đọc dữ liệu...
Đọc dữ liệu thành công!
Đã lấy thêm cột UniqueProducts và Country.
Đã tính toán xong RFM và các biến bổ sung.
Xem trước dữ liệu (đã chuyển vị):
                             0        1        2        3        4
Customer ID            12346.0  12347.0  12348.0  12349.0  12350.0
Recency                    325        2       75       18      310
Frequency                   12        8        5        4        1
Monetary             77,556.46 5,633.32 2,019.40 4,428.69   334.40
AvgOrderValue         6,463.04   704.16   403.88 1,107.17   334.40
UniqueProducts              27      126       25      138       17
TotalItems               74285     3286     2714     1624      197
Country         United Kingdom  Iceland  Finland    Italy   Norway

Đã lưu ../data/processed/customer_features.csv thành công!
